In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

base = expfolder( "system" )[2]

runsim = true;

In [ ]:
# Import parameters.
include( "size-0-params.jl" )

# Import density.
i = 1
ρlist = ρlist[i:i];    Nρ = length( ρlist )
println( "Density: ", ρlist )

# Import speed.
j = Ns
slist = slist[j:j];  Ns = length( slist )
println( "Speed:   ", slist )

# System parameters.
N = 128;  N̂ = 128

# Extract critical noise.
folderdata = [findfolder( N, ρlist[i], slist[j]; base=base, crit=true ) for i ∈ 1:Nρ, j ∈ 1:Ns]
f̂olderdata = [findfolder( N̂, ρlist[i], slist[j]; base=base, crit=true ) for i ∈ 1:Nρ, j ∈ 1:Ns]

# Unpack the kernel parameters.
f0 = 100.0
γdata = first.( readdlm.( folderdata.*"kernel-parameters-power-1.txt" ) )

In [ ]:
# Import area and perimeter scaling values.
βdata = first.( readdlm.( folderdata.*"area-perimeter-scaling.txt" ), 2 )

# Import the cluster change frequency values.
θdata = first.( transpose.( readdlm.( folderdata.*"optimal-rates-hat.txt" ) ), 3 )

# Extract the power law.
λdata = first.( readdlm.( folderdata.*"power-law.txt" ) );

In [ ]:
# Unpack the probability map data.
ℓdata = [polyloglaw.( 1:N̂; β=βdata[i,j] ) for i ∈ 1:Nρ, j ∈ 1:Ns]
Adata = [(ℓdata[i,j].^2)./(4π) for i ∈ 1:Nρ, j ∈ 1:Ns]

# Construct the probability map.
Ωdata = [[runsim ?
    P( k2; N=N̂, i=k1, λℓ=θdata[i,j][1], λr=θdata[i,j][2], κ=θdata[i,j][3] ) :
    0.0
    for k1 in 1:N̂, k2 in 1:N̂] for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Import cluster mass distribution extracted from data.
Fdata = vec.( readdlm.( f̂olderdata.*"cluster-frequency.txt" ) )
fdata = [Fdata[i,j]./sum( Fdata[i,j] ) for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Import kernel coefficient parameters.
αdata = vec.( readdlm.( folderdata.*"kernel-decay-strength.txt" ) )

# Adjust the coagulation-fragmentation ratio for the kernel strength.
γ̂data = f0.*(2.0^first.( αdata )*N̂.^last.( αdata ))

# Initialize parameter variables.
paramsdata = [Params( N̂, Ωdata[i,j], createkernels( N̂, Adata[i,j], ℓdata[i,j], [γ̂data[i,j]] )... )
    for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Simulate the master equations in the larger system size.
T = 10.0
f̂data = [runsim ?
    logsimulate( paramsdata[i,j]; T=T, f=zipf.( 1:N̂; b=N̂, λ=λdata[i,j] ) ) :
    vec( readdlm( f̂olderdata[i,j]*"cluster-distribution-hat.txt" ) )
    for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Extract the power-law of the extrapolated cluster mass distribution.
λ̂data = [
    10^optimize( createzipfloss( N̂, N̂.*f̂data[i,j] ), [log10( 1.5 )] ).minimizer[1]
    for i ∈ 1:Nρ, j ∈ 1:Ns]

# Compute the absolute difference between the distribution exponents.
δλdata = abs.( λdata .- λ̂data )./λdata;
δλmax = 0.01round( maximum( δλdata )/0.01 );

In [ ]:
i = 1;  j = 1
println( "Density: ", ρlist[i] )
println( "Speed:   ", slist[j] )

# Plot bounds.
ιlist = 0.0 .< Fdata[i,j]
xmax = ceil( log10( N̂ ) )
ymin = floor( minimum( log10.( fdata[i,j][ιlist] ) ) )

# Initialize plot variable.
plt = plot(; size=(250,275), legend=true )

# Plot the final distribution from the mean-field equations against the measured.
plot!( plt; legend=:bottomleft, right_margin=0pt )

scatter!( plt, fdata[i,j][ιlist]; color=:gray, markerstrokewidth=0, label=" raw data" )
scatter!( plt, f̂data[i,j][ιlist]; color=:black, markersize=3, markerstrokewidth=0, lw=2, label=" master eq." )

xmax = round( log2( N̂ ) ) + 1
xticks = round.( defInt, 2.0.^(0:2:xmax) )
plot!( plt; xlims=extrema( xticks ), xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt; title="(b)", titleposition=:left )

plot!( plt; framestyle=:box )

plot!( plt; left_margin=0pt, bottom_margin=-5pt, right_margin=10pt )

# saveplot( plt, figurefolder*"master-equations-cmd_N-$(N)-$(N̂)"; background=:white, dpi=600 );

In [ ]:
if false & runsim
    # Save the extrapolated distribution.
    for i ∈ 1:Nρ, j ∈ 1:Ns
        writedlm( f̂olderdata[i,j]*"cluster-distribution-hat.txt", f̂data[i,j] )
    end
end